# Generate an output dataset
Let's see what it's got

## Import batched data

In [ ]:

import json
import pandas as pd
import argparse
from pathlib import Path
import re

from src.utils.helper import extract_messages_from_jsonl


In [ ]:
## If you run your questions as a batch you can stitch them together like this.

msg_10000 = extract_messages_from_jsonl("./data/processed/batch_anthropic_10000_15136.jsonl")
msg_5100 = extract_messages_from_jsonl("./data/processed/batch_anthropic_5100_10000.jsonl")
msg_5000 = extract_messages_from_jsonl("./data/processed/batch_anthropic_5000_5100.jsonl")
msg_0 = extract_messages_from_jsonl("./data/processed/batch_anthropic_5000.jsonl")

msg = pd.concat([msg_0, msg_5000, msg_5100, msg_10000])


In [ ]:
## Basic format changes

msg['event_year'] = pd.to_numeric(msg.event_year, errors='coerce')
# Convert answer to string if not already a list
msg['answer'] = msg['answer'].apply(lambda x: str(x) if not isinstance(x, list) else x)
# Convert answer to list if it's a string
msg['answer'] = msg['answer'].apply(lambda x: [x] if isinstance(x, str) else x)
# Convert subject to list if not already a list
msg['subject'] = msg['subject'].apply(lambda x: [x] if isinstance(x, str) else x)
msg = msg[~msg['answer'].apply(lambda x: isinstance(x, list) and len(x) == 0)]



In [ ]:
# Basic filters
# Find lists containing non-string values in answer column

non_string_lists = msg[msg['answer'].apply(lambda x: isinstance(x, list) and any(not isinstance(item, str) for item in x))]
print("Rows with non-string values in answer lists:")
print(non_string_lists)

# Convert non-string values to strings in answer lists
def convert_to_string_list(x):
    if isinstance(x, list):
        return [str(item) if not isinstance(item, str) else item for item in x]
    elif pd.notnull(x):
        return str(x)
    return x

msg['answer'] = msg['answer'].apply(convert_to_string_list)

# Verify conversion
non_string_lists = msg[msg['answer'].apply(lambda x: isinstance(x, list) and any(not isinstance(item, str) for item in x))]
print("After conversion - Rows with non-string values in answer lists:")
print(non_string_lists)


In [ ]:
# Count messages with parse_error
parse_error_count = len(msg[msg['parse_error'].notna()])
print(f"Number of messages with parse errors: {parse_error_count}")


In [ ]:
import pandas as pd
import pyarrow as pa
# Read in the augmented Wikipedia content
wiki_content = pd.read_parquet("./data/processed/WikipediaContent_augmented.parquet")

# Join msg and wiki_content on id by using the wikipedia_url_id from wiki_content
# Ensure all answer values are lists before merging

# Merge the dataframes
wiki_content_merged = pd.merge(msg, wiki_content,
                    left_on='id',
                    right_on='id', 
                    how='inner')
# Convert back to DataFrame before saving to parquet
wiki_content_merged.to_parquet("./data/processed/temporal_knowledge_wiki_merged.parquet")



## Filter for records we want to keep

Remove anything with less than 6 records, and doesn't have data within the last 5 years.

In [ ]:
import pandas as pd

read_parq = pd.read_parquet("./data/processed/temporal_knowledge_wiki_merged.parquet")
read_parq.count()

In [ ]:
test_msg = read_parq[read_parq.parse_error.isna() & read_parq.event_year.notna()]

invalid_records = test_msg[pd.to_numeric(test_msg.event_year, errors='coerce').isna()].shape[0]
print(f"Number of records with invalid event_year values: {invalid_records}")

# Filter keeping only valid years >= 2000
test_msg['event_year'] = pd.to_numeric(test_msg.event_year, errors='coerce').fillna(0)

test_msg = test_msg[pd.to_numeric(test_msg.event_year, errors='coerce').fillna(0).astype(int) > 1950]

grouped_msg = test_msg.groupby(['id', 'question']).agg(
    distinct_years=('event_year', lambda x: len(set(x[(x >= 2010) & (x <= 2025)]))),
    has_2024=('event_year', lambda x: 2024 in set(x)),
    has_2025=('event_year', lambda x: 2025 in set(x)),
    has_2023=('event_year', lambda x: 2023 in set(x)),
    has_2022=('event_year', lambda x: 2022 in set(x)),
    has_2021=('event_year', lambda x: 2021 in set(x)),
    has_2020=('event_year', lambda x: 2020 in set(x)),
    has_2019=('event_year', lambda x: 2019 in set(x)),
    has_2018=('event_year', lambda x: 2018 in set(x)),
    has_2017=('event_year', lambda x: 2017 in set(x)),
    has_2016=('event_year', lambda x: 2016 in set(x)),
).reset_index()

no_result_pages = grouped_msg[(grouped_msg.distinct_years <= 2)]
result_pages = grouped_msg[(grouped_msg.distinct_years > 6) & ((grouped_msg.has_2024) | (grouped_msg.has_2025) | (grouped_msg.has_2023) | (grouped_msg.has_2022) | (grouped_msg.has_2021) | (grouped_msg.has_2020))]

filtered_parq = read_parq[read_parq['id'].isin(result_pages['id']) & read_parq['question'].isin(result_pages['question'])]

## Clean up duplicate question url and event_year. Find duplicates, for these records, merge the answers into a single row where the answers are a list of strings.
# Merge the answers into a single row where the answers are a list of strings.
## Find duplicates in question, url, event_year
duplicates = filtered_parq[filtered_parq.duplicated(subset=['question', 'id', 'event_year'], keep=False)]

# Merge the answers into a single row where the answers are a list of strings
merged_answers = duplicates.groupby(['question', 'id', 'event_year']).agg({
    'answer': lambda x: list([i for i in x if i is not None])
}).reset_index()

# Flatten the nested lists in the answer column so that the output is a single list of strings
merged_answers['answer'] = merged_answers['answer'].apply(lambda x: [item for sublist in x for item in sublist])


# # assign the answers to the original dataframe, overwrite the answers column
filtered_parq = filtered_parq.merge(merged_answers, on=['question', 'id', 'event_year'], how='left', suffixes=('', '_merged'))
filtered_parq.loc[filtered_parq['answer_merged'].notna(), 'answer'] = filtered_parq['answer_merged']

# # drop the answers_merged column
filtered_parq = filtered_parq.drop(columns=['answer_merged'])

# # drop duplicates
filtered_parq = filtered_parq.drop_duplicates(subset=['question', 'id', 'event_year'])

filtered_parq.to_parquet("./data/processed/data/processed/temporal_knowledge_wiki_merged_filtered.parquet")
filtered_parq.to_csv("./data/processed/temporal_knowledge_wiki_merged_filtered.csv")



## Get pageviews for each article

In [ ]:
import pandas as pd

filtered_parq = pd.read_parquet("./data/processed/temporal_knowledge_wiki_merged_filtered.parquet")
filtered_parq = filtered_parq[['url']].drop_duplicates()
# filtered_parq.head(10)

## Run the file batch_anthropic_5000_concurrent.py to get the pageviews data

In a terminal, or another session that can handle asycio, run the file <br/>
python src/utils/batch_anthropic_5000_concurrent.py

In [ ]:
import pandas as pd

pageviews_df = pd.read_parquet("./data/processed/wikipedia_pageviews_concurrent.parquet")
# Convert timestamp to datetime 
# Try converting to datetime and find problematic records
try:
    # Remove last two zeros from timestamp before converting
    pageviews_df['timestamp'] = pageviews_df['timestamp'].str[:-2]
    pageviews_df['date'] = pd.to_datetime(pageviews_df['timestamp'], format='%Y%m%d')
except ValueError as e:
    # Find records that fail conversion
    mask = pd.to_datetime(pageviews_df['timestamp'], format='%Y%m%d', errors='coerce').isna()
    problematic_records = pageviews_df[mask]
    print(f"Found {len(problematic_records)} problematic records:")
    print(problematic_records)
    raise e

# Extract year from date
pageviews_df['year'] = pageviews_df['date'].dt.year

# Group by wikipedia_url_id and year, sum the views
yearly_views = pageviews_df.groupby(['url', 'year'])['views'].sum().reset_index()

# Sort by url and year
yearly_views = yearly_views.sort_values(['url', 'year'])

print("Yearly pageview statistics:")
print(f"Number of URLs: {len(yearly_views['url'].unique())}")
print(f"Year range: {yearly_views['year'].min()} - {yearly_views['year'].max()}")
print("\nFirst few rows:")
print(yearly_views.head())

# Replace the original dataframe with yearly views
pageviews_df = yearly_views

pageviews_df.info()
# pageviews_df.head(5000)

# Save to CSV and Parquet formats
print("\nSaving files...")

# Save to CSV
csv_path = "./data/processed/wikipedia_pageviews_yearly.csv"
pageviews_df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")

# Save to Parquet 
parquet_path = "./data/processed/wikipedia_pageviews_yearly.parquet"
pageviews_df.to_parquet(parquet_path, index=False)
print(f"Saved Parquet to: {parquet_path}")

print("\nFile sizes:")
import os
print(f"CSV: {os.path.getsize(csv_path)/1024/1024:.1f} MB")
print(f"Parquet: {os.path.getsize(parquet_path)/1024/1024:.1f} MB")


# Join page views with wikicontent

In [ ]:
## Join the pageviews data with the filtered parquet file
import pandas as pd

yearly_views = pd.read_parquet("./data/processed/wikipedia_pageviews_yearly.parquet")
filtered_parq = pd.read_parquet("./data/processed/temporal_knowledge_wiki_merged_filtered.parquet")
# Join the dataframes on url
merged_df = pd.merge(filtered_parq, yearly_views, 
                    left_on=['url', 'event_year'], 
                    right_on=['wikipedia_url_id', 'year'], 
                    how='left')

# Keep only the views column from the right side of the join
merged_df = merged_df.drop(['wikipedia_url_id_y', 'year'], axis=1)
# Rename wikipedia_url_id_x to wikipedia_url_id
merged_df = merged_df.rename(columns={'wikipedia_url_id_x': 'wikipedia_url_id'})

merged_df.to_parquet("./data/processed/temporal_knowledge_wiki_merged_filtered_pageviews.parquet")

# Convert answer and subject columns to string type
# For lists, join elements with comma
merged_df['answer'] = merged_df['answer'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
merged_df['subject'] = merged_df['subject'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))

merged_df.to_csv("./data/processed/temporal_knowledge_wiki_merged_filtered_pageviews.csv", quoting=1)



# Consistency statistics
Understand whether our dataset have a range of homgenous answers over time

In [ ]:
# Group by id and question, then analyze answer similarity within groups
duplicates = merged_df.groupby(['id', 'question']).agg({
    'answer': lambda x: list(x),
    'event_year': lambda x: list(x)
}).reset_index()

# Calculate answer consistency percentages for each group
consistency_stats = []
for _, row in duplicates.iterrows():
    answers = row['answer']
    total_answers = len(answers)
    
    if total_answers > 1:  # Only analyze groups with multiple answers
        # Count frequency of each unique answer
        answer_counts = {}
        for ans in answers:
            answer_counts[ans] = answer_counts.get(ans, 0) + 1
            
        # Find most common answer and its percentage
        most_common_answer = max(answer_counts.items(), key=lambda x: x[1])
        consistency_pct = (most_common_answer[1] / total_answers) * 100
        
        consistency_stats.append({
            'question': row['question'],
            'id': row['id'], 
            'total_answers': total_answers,
            'most_common_answer': most_common_answer[0],
            'consistency_percentage': consistency_pct,
            'unique_answers': list(answer_counts.keys()),
            'years': row['event_year']
        })

# Convert to DataFrame for analysis
consistency_df = pd.DataFrame(consistency_stats)

print(f"Analyzed {len(consistency_df)} question groups with multiple answers")
print("\nExample groups with consistency percentages:")
print(consistency_df[['question', 'consistency_percentage', 'total_answers']].head())

# Calculate overall statistics
avg_consistency = consistency_df['consistency_percentage'].mean()
print(f"\nAverage answer consistency across all groups: {avg_consistency:.2f}%")




In [ ]:
consistency_df[consistency_df['consistency_percentage'] > 80].sort_values(by='consistency_percentage', ascending=False).count()